## ***States and Schemas handling for the Research Assistant***

In [40]:
from typing import Annotated, Literal
# Reuse the application function instead of defining it again here.
from aegis.clarification import check_clarity
from aegis.reducers import merge_by_id

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    StringConstraints,
    model_validator,
)

NonEmptyText = Annotated[
    str,
    StringConstraints(strip_whitespace=True, min_length=1),
]

# Import application code instead of maintaining separate notebook copies.
from aegis.schemas import (
    NonEmptyText,
    ResearchScope,
    ResearchRequest,
    ClarificationQuestion,
    ClarificationResult,
    SourceRecord,
    EvidenceItem,
    ResearchFinding,
    ResearchBundle,
)
from aegis.clarification import check_clarity

# Utilities used by the notebook's examples and validation checks.
from copy import deepcopy
from datetime import datetime, timezone

from pydantic import ValidationError
 

# class ResearchScope(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     time_period: NonEmptyText | None = None
#     coverage: NonEmptyText | None = None


# class ResearchRequest(BaseModel):
#     model_config = ConfigDict(extra="forbid")
#     objective: NonEmptyText
#     # Prototype limits to keep research runs manageable.
#     subjects: list[NonEmptyText] = Field(
#         min_length=1,
#         max_length=5,
#     )
#     # Empty means the factors still need clarification or proposal.
#     factors: list[NonEmptyText] = Field(
#         default_factory=list,
#         max_length=10,
#     )
#     mode: Literal["research", "compare", "rank"] = "research"
#     scope: ResearchScope = Field(default_factory=ResearchScope)
#     constraints: list[NonEmptyText] = Field(default_factory=list)
#     ranking_basis: NonEmptyText | None = None


#     @model_validator(mode="after")
#     def validate_consistency(self):
#         subject_keys = [name.casefold() for name in self.subjects]

#         if len(subject_keys) != len(set(subject_keys)):
#             raise ValueError("Subjects must be unique")

#         factor_keys = [name.casefold() for name in self.factors]

#         if len(factor_keys) != len(set(factor_keys)):
#             raise ValueError("Factors must be unique")

#         if self.mode in {"compare", "rank"} and len(self.subjects) < 2:
#             raise ValueError(
#                 "Comparison or ranking requires at least two subjects"
#             )

#         return self

In [ ]:
# class ClarificationQuestion(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     field: NonEmptyText
#     question: NonEmptyText
#     reason: NonEmptyText


# class ClarificationResult(BaseModel):
#     model_config = ConfigDict(extra="forbid")

#     questions: list[ClarificationQuestion] = Field(
#         default_factory=list
#     )

#     @property
#     def needs_clarification(self) -> bool:
#         return bool(self.questions)

In [ ]:
# def check_clarity(
#     request: ResearchRequest,
# ) -> ClarificationResult:
#     questions = []

#     if not request.factors:
#         questions.append(
#             ClarificationQuestion(
#                 field="factors",
#                 question=(
#                     "Which aspects should the research focus on? "
#                     "You can also ask Aegis to propose some for approval."
#                 ),
#                 reason=(
#                     "The selected factors determine what evidence "
#                     "we need to collect."
#                 ),
#             )
#         )

#     if request.scope.time_period is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="scope.time_period",
#                 question=(
#                     "What period should the research cover? "
#                     "Specify a date range, an as-of date, "
#                     "or say that time is not relevant."
#                 ),
#                 reason=(
#                     "Evidence from different periods may not "
#                     "support a fair comparison."
#                 ),
#             )
#         )

#     if request.scope.coverage is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="scope.coverage",
#                 question=(
#                     "What context or boundaries should apply? "
#                     "For example: competitions, geography, use case, "
#                     "or a general overview."
#                 ),
#                 reason=(
#                     "Coverage determines which evidence belongs "
#                     "in the research."
#                 ),
#             )
#         )

#     if request.mode == "rank" and request.ranking_basis is None:
#         questions.append(
#             ClarificationQuestion(
#                 field="ranking_basis",
#                 question=(
#                     "How should we decide the ranking? "
#                     "State your priorities or provide factor weights."
#                 ),
#                 reason=(
#                     "Choosing a winner requires an explicit "
#                     "decision rule."
#                 ),
#             )
#         )

#     return ClarificationResult(questions=questions)

In [29]:
# Test cell - 1

football_request = ResearchRequest(
    objective=(
        "Compare Lionel Messi and Cristiano Ronaldo "
        "on scoring and creativity."
    ),
    subjects=["Lionel Messi", "Cristiano Ronaldo"],
    factors=["Scoring", "Creativity"],
    mode="compare",
    scope=ResearchScope(
        time_period="2022–2025 calendar years, inclusive",
        coverage="Senior club and international competitive matches",
    ),
    constraints=[
        "Distinguish totals from per-90-minute statistics",
        "Explain differences in competition context",
        "Do not invent unavailable statistics",
        "Prioritize creativity; use scoring as a tiebreaker."
    ],
)

print(football_request.model_dump_json(indent=2))

{
  "objective": "Compare Lionel Messi and Cristiano Ronaldo on scoring and creativity.",
  "subjects": [
    "Lionel Messi",
    "Cristiano Ronaldo"
  ],
  "factors": [
    "Scoring",
    "Creativity"
  ],
  "mode": "compare",
  "scope": {
    "time_period": "2022–2025 calendar years, inclusive",
    "coverage": "Senior club and international competitive matches"
  },
  "constraints": [
    "Distinguish totals from per-90-minute statistics",
    "Explain differences in competition context",
    "Do not invent unavailable statistics",
    "Prioritize creativity; use scoring as a tiebreaker."
  ],
  "ranking_basis": null
}


In [30]:
# Test Cell - 2

platform_request = ResearchRequest(
    objective="Compare data platforms for a small analytics team.",
    subjects=["Snowflake", "Databricks", "Microsoft Fabric"],
    factors=["Operating cost", "Ease of maintenance", "Security"],
    mode="compare",
    scope=ResearchScope(
        coverage="A small team running daily batch analytics",
    ),
)

print(platform_request.model_dump_json(indent=2))

{
  "objective": "Compare data platforms for a small analytics team.",
  "subjects": [
    "Snowflake",
    "Databricks",
    "Microsoft Fabric"
  ],
  "factors": [
    "Operating cost",
    "Ease of maintenance",
    "Security"
  ],
  "mode": "compare",
  "scope": {
    "time_period": null,
    "coverage": "A small team running daily batch analytics"
  },
  "constraints": [],
  "ranking_basis": null
}


In [31]:
# Test Cell - 3 : Failure Cell

from pydantic import ValidationError

try:
    ResearchRequest(
        objective="Compare footballers.",
        subjects=["Lionel Messi"],
        mode="compare",
    )
except ValidationError as exc:
    for error in exc.errors(include_input=False):
        print(error["msg"])
else:
    raise AssertionError("A one-subject comparison was accepted")

Value error, Comparison or ranking requires at least two subjects


In [18]:
incomplete_request = ResearchRequest(
    objective="Compare Messi and Ronaldo.",
    subjects=["Lionel Messi", "Cristiano Ronaldo"],
    mode="compare",
)

result = check_clarity(incomplete_request)

print("Needs clarification:", result.needs_clarification)

for item in result.questions:
    print(f"\nField: {item.field}")
    print(f"Question: {item.question}")
    print(f"Why: {item.reason}")

Needs clarification: True

Field: factors
Question: Which aspects should the research focus on? You can also ask Aegis to propose some for approval.
Why: The selected factors determine what evidence we need to collect.

Field: scope.time_period
Question: What period should the research cover? Specify a date range, an as-of date, or say that time is not relevant.
Why: Evidence from different periods may not support a fair comparison.

Field: scope.coverage
Question: What context or boundaries should apply? For example: competitions, geography, use case, or a general overview.
Why: Coverage determines which evidence belongs in the research.


In [32]:
updated_payload = incomplete_request.model_dump()

updated_payload["factors"] = ["Scoring", "Creativity"]
updated_payload["scope"] = {
    "time_period": "2022–2025 calendar years, inclusive",
    "coverage": (
        "Senior club and international competitive matches"
    ),
}

clarified_request = ResearchRequest.model_validate(updated_payload)

clarified_result = check_clarity(clarified_request)

print("Needs clarification:", clarified_result.needs_clarification)
print("Questions remaining:", len(clarified_result.questions))

Needs clarification: False
Questions remaining: 0


In [33]:
ranking_payload = clarified_request.model_dump()
ranking_payload["mode"] = "rank"

ranking_request = ResearchRequest.model_validate(ranking_payload)
ranking_result = check_clarity(ranking_request)

assert [
    item.field for item in ranking_result.questions
] == ["ranking_basis"]

ranking_payload["ranking_basis"] = (
    "Prioritize creativity; use scoring as a tiebreaker."
)

ranking_request = ResearchRequest.model_validate(ranking_payload)

assert not check_clarity(ranking_request).needs_clarification

print("Comparison and ranking clarification checks passed.")

Comparison and ranking clarification checks passed.


## ***Source | Evidence | Research***

In [ ]:
# from datetime import date, datetime, timezone
# from typing import Literal

# from pydantic import (
#     AwareDatetime,
#     BaseModel,
#     ConfigDict,
#     Field,
#     HttpUrl,
#     ValidationError,
#     model_validator,
# )


# class SourceRecord(BaseModel):
#     """Describe material retrieved by a tool or loaded from a dataset."""

#     model_config = ConfigDict(extra="forbid")

#     # Created by our retrieval code, not invented by the LLM.
#     source_id: NonEmptyText
#     title: NonEmptyText

#     # The source type determines which location field is required.
#     kind: Literal["web", "dataset"]
#     url: HttpUrl | None = None
#     dataset_ref: NonEmptyText | None = None

#     # Unknown publication dates stay unknown.
#     published_on: date | None = None

#     # Explicit timestamp supplied when the material is retrieved.
#     retrieved_at: AwareDatetime

#     # Keeps synthetic examples distinguishable from real research.
#     is_test_data: bool = False

#     @model_validator(mode="after")
#     def validate_location(self):
#         """Require the correct location for the selected source type."""

#         if self.kind == "web":
#             if self.url is None or self.dataset_ref is not None:
#                 raise ValueError(
#                     "Web sources require a URL and no dataset_ref"
#                 )

#         if self.kind == "dataset":
#             if self.dataset_ref is None or self.url is not None:
#                 raise ValueError(
#                     "Dataset sources require dataset_ref and no URL"
#                 )

#         return self

In [ ]:
# class EvidenceItem(BaseModel):
#     """Store an extracted passage or dataset row for later verification."""

#     model_config = ConfigDict(extra="forbid")

#     evidence_id: NonEmptyText

#     # Acts like a foreign key to SourceRecord.source_id.
#     source_id: NonEmptyText

#     # These remain general: a player, product, platform, or another subject.
#     subject: NonEmptyText
#     factor: NonEmptyText

#     # Preserve the retrieved text or a faithful representation of a row.
#     # This is evidence material, not the model's interpretation.
#     excerpt: NonEmptyText

#     # Examples: "Table 2, row 7", "page 4", or a section heading.
#     locator: NonEmptyText

#     # Preserve contextual limits so incompatible statistics aren't mixed.
#     time_period: NonEmptyText | None = None
#     coverage: NonEmptyText | None = None

In [ ]:
# class ResearchFinding(BaseModel):
#     """Represent a sourced claim awaiting independent verification."""

#     model_config = ConfigDict(extra="forbid")

#     finding_id: NonEmptyText
#     subject: NonEmptyText
#     factor: NonEmptyText
#     claim: NonEmptyText

#     # Distinguish a reported observation from an interpretation.
#     claim_type: Literal["reported_fact", "interpretation"]

#     # A finding must reference at least one evidence item.
#     evidence_ids: list[NonEmptyText] = Field(min_length=1)

#     # Record qualifications rather than hiding them in the final report.
#     limitations: list[NonEmptyText] = Field(default_factory=list)

#     # This stage cannot mark its own findings as verified.
#     verification_status: Literal["unverified"] = "unverified"

In [ ]:
# class ResearchBundle(BaseModel):
#     """Group research artifacts and check their reference integrity."""

#     model_config = ConfigDict(extra="forbid")

#     sources: list[SourceRecord] = Field(default_factory=list)
#     evidence: list[EvidenceItem] = Field(default_factory=list)
#     findings: list[ResearchFinding] = Field(default_factory=list)

#     @model_validator(mode="after")
#     def validate_references(self):
#         """Reject duplicate IDs and references to missing records."""

#         source_ids = [item.source_id for item in self.sources]
#         evidence_ids = [item.evidence_id for item in self.evidence]
#         finding_ids = [item.finding_id for item in self.findings]

#         # Duplicate IDs would make citation resolution ambiguous.
#         for label, identifiers in (
#             ("source", source_ids),
#             ("evidence", evidence_ids),
#             ("finding", finding_ids),
#         ):
#             if len(identifiers) != len(set(identifiers)):
#                 raise ValueError(f"Duplicate {label} IDs")

#         known_sources = set(source_ids)
#         known_evidence = set(evidence_ids)

#         # Every evidence item must point to an existing source.
#         for item in self.evidence:
#             if item.source_id not in known_sources:
#                 raise ValueError(
#                     f"Evidence {item.evidence_id} references "
#                     f"unknown source {item.source_id}"
#                 )

#         # Every finding must point to existing, distinct evidence items.
#         for finding in self.findings:
#             if len(finding.evidence_ids) != len(set(finding.evidence_ids)):
#                 raise ValueError(
#                     f"Finding {finding.finding_id} repeats evidence IDs"
#                 )

#             missing = set(finding.evidence_ids) - known_evidence

#             if missing:
#                 raise ValueError(
#                     f"Finding {finding.finding_id} references "
#                     f"unknown evidence: {sorted(missing)}"
#                 )

#         return self

In [34]:
#  Test Cell - 4

# Create a synthetic source record.
# No file is read here; the dataset reference is just a test identifier.
source = SourceRecord(
    source_id="src_demo_001",
    title="Synthetic football statistics for schema testing",
    kind="dataset",
    dataset_ref="fixture://football_demo_v1",
    retrieved_at=datetime.now(timezone.utc),
    is_test_data=True,
)

# Represent one row from that fictional dataset.
evidence = EvidenceItem(
    evidence_id="ev_demo_001",
    source_id=source.source_id,
    subject="Player A",
    factor="Scoring",
    excerpt="Player A | appearances: 10 | goals: 8",
    locator="players table, row player_a",
    time_period="Demo season",
    coverage="Fictional competition",
)

# Make a reported claim supported by that row.
finding = ResearchFinding(
    finding_id="finding_demo_001",
    subject="Player A",
    factor="Scoring",
    claim="The demo dataset records 8 goals for Player A.",
    claim_type="reported_fact",
    evidence_ids=[evidence.evidence_id],
    limitations=["Synthetic data; not suitable for real comparisons."],
)

# Validate all objects and their links together.
bundle = ResearchBundle(
    sources=[source],
    evidence=[evidence],
    findings=[finding],
)

print(bundle.model_dump_json(indent=2))

{
  "sources": [
    {
      "source_id": "src_demo_001",
      "title": "Synthetic football statistics for schema testing",
      "kind": "dataset",
      "url": null,
      "dataset_ref": "fixture://football_demo_v1",
      "published_on": null,
      "retrieved_at": "2026-11-17T16:07:44.052507Z",
      "is_test_data": true
    }
  ],
  "evidence": [
    {
      "evidence_id": "ev_demo_001",
      "source_id": "src_demo_001",
      "subject": "Player A",
      "factor": "Scoring",
      "excerpt": "Player A | appearances: 10 | goals: 8",
      "locator": "players table, row player_a",
      "time_period": "Demo season",
      "coverage": "Fictional competition"
    }
  ],
  "findings": [
    {
      "finding_id": "finding_demo_001",
      "subject": "Player A",
      "factor": "Scoring",
      "claim": "The demo dataset records 8 goals for Player A.",
      "claim_type": "reported_fact",
      "evidence_ids": [
        "ev_demo_001"
      ],
      "limitations": [
        "Synthetic 

In [35]:
# Reject broken records

from copy import deepcopy

# Export a valid bundle into ordinary Python data.
valid_payload = bundle.model_dump(mode="json")


def expect_rejection(payload: dict, expected_message: str) -> None:
    """Confirm that invalid research data fails for the expected reason."""

    try:
        ResearchBundle.model_validate(payload)
    except ValidationError as exc:
        messages = " | ".join(
            error["msg"] for error in exc.errors(include_input=False)
        )
        assert expected_message in messages, messages
        print(f"PASS: {expected_message}")
    else:
        raise AssertionError("Invalid research data was accepted")


# Test 1: Evidence points to a source that doesn't exist.
bad_source = deepcopy(valid_payload)
bad_source["evidence"][0]["source_id"] = "src_missing"
expect_rejection(bad_source, "unknown source")


# Test 2: A finding cites evidence that doesn't exist.
bad_citation = deepcopy(valid_payload)
bad_citation["findings"][0]["evidence_ids"] = ["ev_missing"]
expect_rejection(bad_citation, "unknown evidence")


# Test 3: Two sources share an ID, making references ambiguous.
duplicate_source = deepcopy(valid_payload)
duplicate_source["sources"].append(
    deepcopy(duplicate_source["sources"][0])
)
expect_rejection(duplicate_source, "Duplicate source IDs")

PASS: unknown source
PASS: unknown evidence
PASS: Duplicate source IDs


## ***Graph state and reducers***

Graph state contains the shared information for one research run.
Each node returns only the fields it wants to update.

A reducer defines how an incoming update combines with an existing value.

The owners below describe our intended architecture. Most of these nodes
will be implemented later.

| State key | Intended writer | Value type | Update rule |
|---|---|---|---|
| request | Input/clarification boundary | ResearchRequest | Replace |
| clarification | Clarification node | ClarificationResult or None | Replace |
| plan | Planner | List of task records | Replace |
| sources | Evidence workers | Dictionary keyed by source ID | Merge by ID |
| evidence | Evidence workers | Dictionary keyed by evidence ID | Merge by ID |
| findings | Research workers | Dictionary keyed by finding ID | Merge by ID |
| calculations | Analysis workers | Dictionary keyed by calculation ID | Merge by ID |
| verification | Verification coordinator | Verification result or None | Replace |
| approval | Approval boundary | Approval decision or None | Replace |
| report | Reporter | Final report or None | Replace |
| status | Active coordinating node | RunStatus | Replace |
| retry_count | Retry coordinator | Integer | Replace |
| usage | Call-tracking integration | Usage grouped by provider/model | Add deltas |
| errors | Any node handling an error | List of error records | Append |
| node_events | Any instrumented node | List of event records | Append |

Fields using replacement must have at most one writer in a graph step.
Parallel workers return their own updates, not copies of the entire state.

In [36]:
from typing import TypedDict

from langgraph.errors import InvalidUpdateError
from langgraph.graph import START, END, StateGraph


class UnmergedState(TypedDict):
    """Minimal state with no rule for combining evidence updates."""

    evidence: dict[str, str]


def worker_a_without_reducer(
    state: UnmergedState,
) -> dict:
    """Return only Worker A's new evidence."""

    return {
        "evidence": {
            "ev_a": "Synthetic evidence about Player A",
        }
    }


def worker_b_without_reducer(
    state: UnmergedState,
) -> dict:
    """Return only Worker B's new evidence."""

    return {
        "evidence": {
            "ev_b": "Synthetic evidence about Player B",
        }
    }


def join_without_reducer(
    state: UnmergedState,
) -> dict:
    """Read the combined evidence after both workers finish."""

    print("Evidence received by join:", state["evidence"])

    # An empty update means this node changes nothing.
    return {}


# Tell LangGraph which state definition this graph uses.
unmerged_builder = StateGraph(UnmergedState)

# Register the functions as graph nodes.
unmerged_builder.add_node("worker_a", worker_a_without_reducer)
unmerged_builder.add_node("worker_b", worker_b_without_reducer)
unmerged_builder.add_node("join", join_without_reducer)

# Both workers start in the same graph step.
unmerged_builder.add_edge(START, "worker_a")
unmerged_builder.add_edge(START, "worker_b")

# The list-form edge makes join wait for BOTH workers.
unmerged_builder.add_edge(["worker_a", "worker_b"], "join")
unmerged_builder.add_edge("join", END)

# Compile the graph into something we can invoke.
unmerged_graph = unmerged_builder.compile()


# This failure is intentional: both workers update evidence,
# but the state has no reducer for combining those updates.
try:
    unmerged_graph.invoke({"evidence": {}})

except InvalidUpdateError:
    print(
        "PASS: parallel evidence updates were rejected "
        "because no reducer is defined."
    )

else:
    raise AssertionError(
        "Expected parallel writes without a reducer to fail."
    )

PASS: parallel evidence updates were rejected because no reducer is defined.


In [37]:
from typing import TypeVar


# T allows this function to work with different record types:
# strings now, and SourceRecord/EvidenceItem objects later.
T = TypeVar("T")


def merge_by_id(
    current: dict[str, T],
    update: dict[str, T],
) -> dict[str, T]:
    """Combine records, replacing an existing record if its ID repeats."""

    # Build a new dictionary without changing either input.
    # Because update comes last, its values replace matching IDs.
    return {**current, **update}

In [38]:
from typing import Annotated


class MergedState(TypedDict):
    """State that knows how to combine evidence from multiple workers."""

    # Annotated attaches the reducer to this state field.
    # LangGraph calls merge_by_id when evidence updates arrive.
    evidence: Annotated[dict[str, str], merge_by_id]


def worker_a(state: MergedState) -> dict:
    """Produce one synthetic evidence record."""

    return {
        "evidence": {
            "ev_a": "Synthetic evidence about Player A",
        }
    }


def worker_b(state: MergedState) -> dict:
    """Produce a different synthetic evidence record."""

    return {
        "evidence": {
            "ev_b": "Synthetic evidence about Player B",
        }
    }


def join(state: MergedState) -> dict:
    """Check that both workers' records survived the merge."""

    assert set(state["evidence"]) == {"ev_a", "ev_b"}

    print("PASS: join received evidence from both workers")

    return {}


merged_builder = StateGraph(MergedState)

merged_builder.add_node("worker_a", worker_a)
merged_builder.add_node("worker_b", worker_b)
merged_builder.add_node("join", join)

merged_builder.add_edge(START, "worker_a")
merged_builder.add_edge(START, "worker_b")

merged_builder.add_edge(["worker_a", "worker_b"], "join")
merged_builder.add_edge("join", END)

merged_graph = merged_builder.compile()

merged_result = merged_graph.invoke({"evidence": {}})

print(merged_result["evidence"])

PASS: join received evidence from both workers
{'ev_a': 'Synthetic evidence about Player A', 'ev_b': 'Synthetic evidence about Player B'}


In [39]:
def retry_worker_a(state: MergedState) -> dict:
    """Simulate a later correction to Worker A's evidence."""

    # Keep the same ID so the reducer replaces the earlier record.
    return {
        "evidence": {
            "ev_a": "Corrected synthetic evidence about Player A",
        }
    }


retry_builder = StateGraph(MergedState)

retry_builder.add_node("worker_a", worker_a)
retry_builder.add_node("worker_b", worker_b)
retry_builder.add_node("join", join)
retry_builder.add_node("retry_worker_a", retry_worker_a)

# First, run both original workers in parallel.
retry_builder.add_edge(START, "worker_a")
retry_builder.add_edge(START, "worker_b")

# Wait for both workers before applying a correction.
retry_builder.add_edge(["worker_a", "worker_b"], "join")
retry_builder.add_edge("join", "retry_worker_a")
retry_builder.add_edge("retry_worker_a", END)

retry_graph = retry_builder.compile()
retry_result = retry_graph.invoke({"evidence": {}})

# The correction replaces ev_a instead of creating a third record.
assert len(retry_result["evidence"]) == 2

assert retry_result["evidence"]["ev_a"] == (
    "Corrected synthetic evidence about Player A"
)

# Worker B's evidence must remain unchanged.
assert retry_result["evidence"]["ev_b"] == (
    "Synthetic evidence about Player B"
)

print("PASS: retry replaced ev_a without duplication")
print("PASS: ev_b remained unchanged")

PASS: join received evidence from both workers
PASS: retry replaced ev_a without duplication
PASS: ev_b remained unchanged


In [41]:
from datetime import datetime, timezone

from langgraph.graph import START, END, StateGraph

from aegis.schemas import EvidenceItem, SourceRecord
from aegis.state import (
    AegisState,
    RunStatus,
    bundle_from_state,
    make_evidence_id,
)


# This source is a synthetic fixture identifier.
# No external file or API is accessed.
demo_source = SourceRecord(
    source_id="src_state_demo_v1",
    title="Synthetic evidence for graph-state experiments",
    kind="dataset",
    dataset_ref="fixture://state_demo_v1",
    retrieved_at=datetime(2026, 1, 1, tzinfo=timezone.utc),
    is_test_data=True,
)


def make_demo_evidence(
    subject: str,
    goals: int,
) -> EvidenceItem:
    """Create a validated fictional observation."""

    locator = f"players/{subject}"

    return EvidenceItem(
        evidence_id=make_evidence_id(
            demo_source.source_id,
            locator,
            subject,
            "Scoring",
            time_period="Demo season",
            coverage="Fictional competition",
        ),
        source_id=demo_source.source_id,
        subject=subject,
        factor="Scoring",
        excerpt=f"Synthetic goals: {goals}",
        locator=locator,
        time_period="Demo season",
        coverage="Fictional competition",
    )


def research_a(state: AegisState) -> dict:
    """Emit only this worker's evidence and event."""

    record = make_demo_evidence("Player A", 8)

    return {
        "evidence": {record.evidence_id: record},
        "node_events": [{"node": "research_a", "attempt": 1}],
    }


def research_b(state: AegisState) -> dict:
    """Contribute a different observation in parallel."""

    record = make_demo_evidence("Player B", 6)

    return {
        "evidence": {record.evidence_id: record},
        "node_events": [{"node": "research_b", "attempt": 1}],
    }


def validate_join(state: AegisState) -> dict:
    """Validate references only after both workers have finished."""

    bundle = bundle_from_state(state)

    assert len(bundle.evidence) == 2
    print("PASS: joined evidence passes bundle validation")

    return {}


def correct_a(state: AegisState) -> dict:
    """Update Player A's content while retaining its evidence ID."""

    corrected = make_demo_evidence("Player A", 9)

    return {
        "evidence": {corrected.evidence_id: corrected},
        "retry_count": state.get("retry_count", 0) + 1,
        "node_events": [{"node": "research_a", "attempt": 2}],
    }


def finish_demo(state: AegisState) -> dict:
    """Verify the correction before completing this toy workflow."""

    bundle = bundle_from_state(state)

    by_subject = {
        record.subject: record
        for record in bundle.evidence
    }

    assert len(bundle.evidence) == 2
    assert by_subject["Player A"].excerpt == "Synthetic goals: 9"
    assert by_subject["Player B"].excerpt == "Synthetic goals: 6"

    return {"status": RunStatus.COMPLETED}


builder = StateGraph(AegisState)

builder.add_node("research_a", research_a)
builder.add_node("research_b", research_b)
builder.add_node("validate_join", validate_join)
builder.add_node("correct_a", correct_a)
builder.add_node("finish_demo", finish_demo)

builder.add_edge(START, "research_a")
builder.add_edge(START, "research_b")
builder.add_edge(["research_a", "research_b"], "validate_join")
builder.add_edge("validate_join", "correct_a")
builder.add_edge("correct_a", "finish_demo")
builder.add_edge("finish_demo", END)

state_demo_graph = builder.compile()

state_demo_result = state_demo_graph.invoke(
    {
        "sources": {demo_source.source_id: demo_source},
        "status": RunStatus.RUNNING,
        "retry_count": 0,
    }
)

assert state_demo_result["retry_count"] == 1
assert len(state_demo_result["node_events"]) == 3
assert state_demo_result["status"] == RunStatus.COMPLETED

print("PASS: corrected evidence replaced the original")
print("PASS: all three worker-attempt events were retained")

PASS: joined evidence passes bundle validation
PASS: corrected evidence replaced the original
PASS: all three worker-attempt events were retained


In [42]:
from decimal import Decimal

from pydantic import ValidationError

from aegis.reducers import sum_usage


# Fictional usage deltas: these do not represent actual API calls.
first_call = {
    "demo/model": {
        "tokens_in": 100,
        "tokens_out": 20,
        "cost_usd": Decimal("0.001"),
    }
}

second_call = {
    "demo/model": {
        "tokens_in": 50,
        "tokens_out": 10,
        "cost_usd": Decimal("0.002"),
    }
}

combined_usage = sum_usage(first_call, second_call)

assert combined_usage["demo/model"] == {
    "tokens_in": 150,
    "tokens_out": 30,
    "cost_usd": Decimal("0.003"),
}

print("PASS: usage deltas were added correctly")


# Remove the source while retaining evidence that references it.
# Build a separate state so the successful result stays unchanged.
broken_state = {
    **state_demo_result,
    "sources": {},
}

try:
    bundle_from_state(broken_state)

except ValidationError as exc:
    assert "unknown source" in str(exc).lower()
    print("PASS: broken source reference was rejected")

else:
    raise AssertionError("A broken source reference was accepted")

PASS: usage deltas were added correctly
PASS: broken source reference was rejected
